In [20]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

In [21]:
df=pd.read_csv('future_jobs_dataset.csv')

In [22]:
df.head()

,job_id,job_title,industry,location,salary_usd,skills_required,remote_option,company_size,posting_date
0,1,Quantum Researcher,Quantum Computing,Singapore,175780,"Linear Algebra, Quantum Algorithms",No,Large,2025-07-22
1,2,Renewable Energy Engineer,Green Tech,Singapore,137481,"Climate Data Analysis, Energy Modeling",Yes,Large,2025-09-26
2,3,Quantum Researcher,Quantum Computing,Tokyo,182081,"Linear Algebra, Qiskit",No,Medium,2025-12-31
3,4,Sustainability Analyst,Green Tech,Singapore,113822,"Climate Data Analysis, Energy Modeling",No,Large,2025-05-29
4,5,Smart Contract Engineer,Blockchain,London,92575,"Rust, Solidity",Yes,Small,2025-03-30


In [23]:
df.tail()

,job_id,job_title,industry,location,salary_usd,skills_required,remote_option,company_size,posting_date
9995,9996,Sustainability Analyst,Green Tech,New York,248617,"Energy Modeling, Climate Data Analysis",Yes,Small,2025-04-13
9996,9997,Blockchain Developer,Blockchain,New York,104410,"Ethereum, Solidity",Yes,Large,2025-11-03
9997,9998,Quantum Software Developer,Quantum Computing,Berlin,200382,"Quantum Algorithms, Qiskit",No,Small,2025-02-23
9998,9999,Renewable Energy Engineer,Green Tech,New York,214484,"Energy Modeling, Climate Data Analysis",No,Small,2025-03-08
9999,10000,AI Engineer,AI,Singapore,117307,"TensorFlow, Python",Yes,Large,2025-08-12


In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   job_id           10000 non-null  int64 
 1   job_title        10000 non-null  object
 2   industry         10000 non-null  object
 3   location         10000 non-null  object
 4   salary_usd       10000 non-null  int64 
 5   skills_required  10000 non-null  object
 6   remote_option    10000 non-null  object
 7   company_size     10000 non-null  object
 8   posting_date     10000 non-null  object
dtypes: int64(2), object(7)
memory usage: 703.2+ KB


In [25]:
df.columns

Index(['job_id', 'job_title', 'industry', 'location', 'salary_usd',
       'skills_required', 'remote_option', 'company_size', 'posting_date'],
      dtype='object')

In [26]:
df['posting_date'] = pd.to_datetime(df['posting_date'])

In [27]:
salary_by_role = df.groupby("job_title")["salary_usd"].mean().sort_values(ascending=False)

print(salary_by_role.head(10))

job_title
Quantum Software Developer    151499.841941
ML Researcher                 151284.985222
Sustainability Analyst        150991.592563
Data Scientist                150990.363525
Smart Contract Engineer       150779.203187
Quantum Researcher            150750.544722
AI Engineer                   149794.911935
Blockchain Developer          147975.987942
Renewable Energy Engineer     147689.426177
Name: salary_usd, dtype: float64


In [28]:
recent_jobs = df[df['posting_date'] >= df['posting_date'].max() - pd.Timedelta(days=90)]

industry_growth = recent_jobs['industry'].value_counts()

print(industry_growth.head(10))

Quantum Computing    657
Blockchain           633
AI                   626
Green Tech           619
Name: industry, dtype: int64


In [29]:
from collections import Counter

all_skills = []

for skills in df['skills_required'].dropna():
    skill_list = [s.strip() for s in skills.split(",")]
    all_skills.extend(skill_list)

skill_counts = Counter(all_skills)

top_skills = pd.DataFrame(skill_counts.items(), columns=["Skill", "Count"])
top_skills = top_skills.sort_values("Count", ascending=False)

print(top_skills.head(15))

                    Skill  Count
2   Climate Data Analysis   2490
3         Energy Modeling   2490
1      Quantum Algorithms   1690
8              TensorFlow   1690
7                Ethereum   1683
4                  Qiskit   1682
6                Solidity   1669
0          Linear Algebra   1666
10                 Python   1657
5                    Rust   1646
9                 PyTorch   1637


In [30]:
location_salary = df.groupby("location")["salary_usd"].mean().sort_values(ascending=False)

print(location_salary.head(10))

location
New York     152266.601539
London       152260.956522
Berlin       150194.844512
Dubai        149483.852410
Singapore    148622.504162
Tokyo        147965.942618
Name: salary_usd, dtype: float64


In [31]:
remote_ratio = df.groupby("job_title")["remote_option"].value_counts(normalize=True)

print(remote_ratio)

job_title                   remote_option
AI Engineer                 No               0.541136
                            Yes              0.458864
Blockchain Developer        Yes              0.516077
                            No               0.483923
Data Scientist              Yes              0.526316
                            No               0.473684
ML Researcher               Yes              0.523399
                            No               0.476601
Quantum Researcher          Yes              0.509267
                            No               0.490733
Quantum Software Developer  Yes              0.508607
                            No               0.491393
Renewable Energy Engineer   Yes              0.509178
                            No               0.490822
Smart Contract Engineer     Yes              0.510757
                            No               0.489243
Sustainability Analyst      Yes              0.513339
                            No          

In [32]:
company_salary = df.groupby("company_size")["salary_usd"].mean()

print(company_salary)

company_size
Large     150876.260561
Medium    150188.150541
Small     149307.253118
Name: salary_usd, dtype: float64


In [33]:
df['career_score'] = (
    df['salary_usd'] * 0.5 +
    df['remote_option'].map({'Yes':1,'No':0}) * 10000 +
    df['job_title'].map(salary_by_role) * 0.2
)

best_jobs = df.sort_values("career_score", ascending=False)

best_jobs[['job_title','location','salary_usd','remote_option']].head(10)

,job_title,location,salary_usd,remote_option
5076,Quantum Software Developer,Tokyo,249766,Yes
78,Data Scientist,Dubai,249952,Yes
2757,Smart Contract Engineer,Singapore,249985,Yes
1647,Smart Contract Engineer,Tokyo,249984,Yes
4863,Quantum Software Developer,Berlin,249634,Yes
7170,Smart Contract Engineer,Tokyo,249846,Yes
8076,Smart Contract Engineer,London,249784,Yes
9789,Data Scientist,Berlin,249608,Yes
5666,Data Scientist,Berlin,249546,Yes
4129,Quantum Researcher,New York,249626,Yes


In [34]:
target_jobs = df[
    (df['salary_usd'] > df['salary_usd'].median()) &
    (df['remote_option'] == 'Yes')
]

target_jobs.head()

,job_id,job_title,industry,location,salary_usd,skills_required,remote_option,company_size,posting_date,career_score
5,6,Smart Contract Engineer,Blockchain,Tokyo,173379,"Solidity, Rust",Yes,Medium,2025-08-10,126845.340637
7,8,Quantum Software Developer,Quantum Computing,London,210842,"Qiskit, Quantum Algorithms",Yes,Large,2025-04-13,145720.968388
16,17,Renewable Energy Engineer,Green Tech,London,214393,"Climate Data Analysis, Energy Modeling",Yes,Small,2025-04-11,146734.385235
17,18,Data Scientist,AI,Singapore,190743,"TensorFlow, Python",Yes,Small,2025-10-07,135569.572705
20,21,Smart Contract Engineer,Blockchain,Tokyo,180309,"Solidity, Ethereum",Yes,Small,2025-11-23,130310.340637


In [35]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(df['skills_required'].fillna(""))

In [36]:
my_profile = "Python SQL Power BI"

my_vector = vectorizer.transform([my_profile])

similarity_scores = cosine_similarity(my_vector, tfidf_matrix)

df['similarity_score'] = similarity_scores[0]

In [37]:
recommended_jobs = df.sort_values("similarity_score", ascending=False)

recommended_jobs[['job_title','location','salary_usd','similarity_score']].head(10)

,job_title,location,salary_usd,similarity_score
9999,AI Engineer,Singapore,117307,0.709602
6281,AI Engineer,London,62462,0.709602
4437,Data Scientist,London,199279,0.709602
4438,Data Scientist,Dubai,200484,0.709602
4440,ML Researcher,Berlin,224777,0.709602
1446,Data Scientist,Berlin,228324,0.709602
1444,Data Scientist,Berlin,222961,0.709602
1443,ML Researcher,Dubai,173103,0.709602
4442,Data Scientist,Berlin,176256,0.709602
6284,Data Scientist,Tokyo,111781,0.709602
